## 数据回放测试，用于查找问题

In [ ]:
from chanlun.backtesting.backtest_klines import BackTestKlines
from chanlun import kcharts
from chanlun.cl_utils import query_cl_chart_config
from chanlun.exchange.exchange import *
import pandas as pd
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import ttk, messagebox
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from chanlun.exchange.exchange_db import ExchangeDB
import time
from chanlun import cl
import os
import webbrowser
from chanlun.backtesting.base import Strategy

In [ ]:

import pandas as pd
from chanlun.cl_utils import query_cl_chart_config
from chanlun.exchange.exchange_db import ExchangeDB
from chanlun.cl_utils import *
def get_history(
    market: str = 'a',
    code: str = 'SZ.002941',
    frequency: str = 'd',
    count: int = 0,
    ealry: bool = False, 
    start_date: str = "2019-05-16 09:30:00",
    end_date: str = "2024-08-22 15:01:00"
) -> pd.DataFrame:
    """
    获取历史K线数据并进行预处理
    
    参数:
        market (str): 市场标识符，默认为 'a' 表示沪深A股。
        code (str): 标的资产代码，默认为 'SHSE.000300'。
        frequency (str): 周期列表，默认为 '1m'。
        start_date (str): 开始日期和时间，默认为 "2025-05-16 09:30:00"。
        end_date (str): 结束日期和时间，默认为 "2025-05-16 15:01:00"。
        
    返回:
        pd.DataFrame: 包含 'open', 'high', 'low', 'close', 'volume' 的 DataFrame，索引为 datetime。
    """
    
    # 查询缠论配置
    cl_config = query_cl_chart_config(market, code)
    # print(cl_config)

    # 使用 ExchangeDB 读取数据库中的 K 线数据
    ex = ExchangeDB(market)
    # print("end_date",end_date)
    # 获取 K 线数据
    klines = ex.klines(code, frequency=frequency, start_date=start_date, end_date=end_date)
    # print(klines.iloc[-1])
    # 提取所需列并确保 'date' 列是 datetime 类型
    if count and not ealry:
        df = klines[['date', 'open', 'high', 'low', 'close', 'volume']][-count:]
    elif count and ealry:
        df = klines[['date', 'open', 'high', 'low', 'close', 'volume']][:count]
    else:
        df = klines[['date', 'open', 'high', 'low', 'close', 'volume']]
    df['date'] = pd.to_datetime(df['date'])

    # 设置 'date' 列为索引
    df.set_index('date', inplace=True)

    # 添加 timestamp 列（可选）
    df['timestamp'] = df.index

    return df

if __name__ == '__main__':
    df = get_history(market='a', code='SH.600031',count=300, frequency='5m', start_date="2025-05-16 09:30:00", end_date="2025-09-05 10:01:00")
    print(df.tail(100))

In [ ]:
market = 'a'
code = 'SH.600262'
start_date = '2025-03-01 09:30:00'
end_date = '2025-11-17 15:01:00'
frequencys = ["d","30m","5m"]
ex = ExchangeDB(market) # 读取数据库中的k线数据
cl_config = query_cl_chart_config(market, code) # 读取缠论配置
# cl_config["to_file"] = f"E:\stock_history_charts\{code}_{frequencys[0]}_{start_date[:10]}_{end_date[:10]}.html"

In [ ]:
print(cl_config)

In [ ]:
from chanlun import fun
import talib as ta
import numpy as np
cl_config = query_cl_chart_config(market, code)
neck_price = 24.8
stop_atr = 0.71

info = {}
bk = BackTestKlines(market, start_date, end_date, frequencys, cl_config)
bk.init(code, frequencys[-1])
# 依次执行此单元格，检查缠论计算结果
while (bk.next()):
    if bk.now_date <= fun.str_to_datetime("2025-08-16 09:30:00"):
        continue            
    low_cd = bk.get_cl_data(code, "5m")
    klines = low_cd.get_src_klines()
    last_k = klines[-1]
    pre_k = klines[-2]
    kdate = last_k.date
    amounts = np.array([_k.a for _k in klines])
    ma20_amount = ta.MA(amounts, 20)


    # 小级别放量突破中级别颈线位，买入做多
    if (
        last_k.c > neck_price
        and pre_k.c <= neck_price 
        and last_k.a > pre_k.a * 3
        and last_k.a > ma20_amount[-2] * 3
        and (last_k.h - last_k.c) / (last_k.h - pre_k.c) < 0.5
    ):
        #上引线占比
        info['a_by_pre'] = last_k.a / pre_k.a
        info['a_by_ma20'] = last_k.a / ma20_amount[-2]
        k_value = (last_k.h - last_k.c) / (last_k.h - pre_k.c)
        info["k_value"] = k_value
        loss_price = min(pre_k.l,last_k.l)  - stop_atr * 1.5
        info["open_k_date"] = kdate
        print(f"{kdate}收盘价{last_k.c}突破{neck_price}买入")
        


    low_cd = bk.get_cl_data(code, "5m")
    low_xds = low_cd.get_xds()
    last_xd = low_xds[-1]
    for i in range(len(low_xds) - 1,0,-1):
        if low_xds[i].start.k.date <= fun.str_to_datetime("2025-08-25 09:30:00"):
            break
    last_xds = low_xds[i:]
    if len(last_xds)< 3 or last_xd.type == "down":
        continue
    low_bis = low_cd.get_bis()
    last_bi = low_bis[-1]
    if last_xds[-1].high < last_xds[-2].high and last_bi.type == "down" and last_bi.is_done():
        print(f"{kdate}以{last_k.c}卖出")
        print(info)
        # kcharts.render_charts(code, low_cd, config=cl_config)

        break


In [ ]:
kcharts.render_charts(code, low_cd, config=cl_config)

In [ ]:
# while (bk.next()):
#     if bk.now_date <= fun.str_to_datetime('2025-07-08 08:45:00'):
#         continue
#     else:
#         break
bk.next(frequency="5m")
cd = bk.get_cl_data(code, "5m")
kcharts.render_charts(code, cd, config=cl_config)

In [ ]:
bk.now_date

In [ ]:
# 回放的数据配置
# market = 'currency'
# code = 'ETH/USDT'
# start_date = '2023-02-02 14:30:00'
# end_date = '2023-02-02 15:00:00'
# frequencys = ['5m']



cl_config = query_cl_chart_config(market, code)

bk = BackTestKlines(market, start_date, end_date, frequencys, cl_config)
bk.init(code, frequencys[0])

In [ ]:
# 依次执行此单元格，检查缠论计算结果
while (bk.next()):
    cd = bk.get_cl_data(code, frequencys[0])
    # kcharts.render_charts(code, cd, config=cl_config)
    high_cd = cd
    klines_lv0 = high_cd.get_klines()
    last_k = high_cd.get_src_klines()[-1]
    pre_k = high_cd.get_src_klines()[-2]
    price = last_k.c
    xdzs = high_cd.get_last_xd_zs()

    xd_start = xdzs.lines[0]
    xd_zs_klines_lv0 = klines_lv0[xd_start.start.k.k_index:]
    zz_xd,zd_xd,zg_xd = Strategy.get_frvp(cd_klines=xd_zs_klines_lv0, use_close=False)
    bi = high_cd.get_bis()[-1]
    pre_bi = high_cd.get_bis()[-2]
    # 触发信号
    if (
        bi.type == "down"
        and bi.end.done
        and abs(bi.low - zg_xd) <= zg_xd * 0.0005
    ):
        # print(f"{xd_start},{xd_zs_klines_lv0[0]}")
        # print(f"{last_k.date}触发,{xd_start.start.k.index},{xd_zs_klines_lv0[0].date},{xd_zs_klines_lv0[-1].date},{price},{bi.low}")
        # 入场信号
        frvp_bi_start_index = pre_bi.start.k.k_index
        print(f"index： {frvp_bi_start_index}")
        pos_info = {}
        pos_info["frvp_bi_start_index"] = frvp_bi_start_index
        bi_zs_klines_lv0 = klines_lv0[frvp_bi_start_index:]
        print(pre_bi)
        print(bi_zs_klines_lv0[0])
        print(bi_zs_klines_lv0[-1])
        zz_bi,zd_bi,zg_bi = Strategy.get_frvp(cd_klines=bi_zs_klines_lv0, use_close=False)

In [ ]:
cd = bk.get_cl_data(code, frequencys[0])
kcharts.render_charts(code, cd, config=cl_config)

In [ ]:
klines = cd2.get_src_klines()
k1 = klines[-1]
k2 = klines[-2]
print(k1.date,k2.date,(k2.date-k1.date).days)
# datetime.(k2.date,k1.date)

In [ ]:
klines

In [ ]:
zss = cd.get_xd_zss()
zs = zss[-1]
print(zs)

In [ ]:
last_zs = cd.get_last_xd_zs()
print(last_zs)
for line in last_zs.lines:
    print(line)

In [ ]:
print(zs.type)

In [ ]:
print(cd.get_last_xd_zs())

In [ ]:
for line in zs.lines:
    print(line)

In [ ]:
bk.now_date

In [ ]:
print(cd.get_xds()[-1])